# 🏀 Object Tracking in Sports (Basketball Example)
This notebook demonstrates how to use OpenCV's tracking algorithms (like CSRT) to track a player or ball in a basketball match video.

In [7]:
import cv2
import numpy as np
import os
from IPython.display import Video, display

video_path = '../data/basketball_clip2.mp4'
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise IOError("Failed to open video")

success, frame = cap.read()
if not success or frame is None:
    raise IOError("Couldn't read the first frame of the video")

# Convert first frame to HSV
hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

# Define HSV range for basketball orange
lower_orange = np.array([5, 100, 100])
upper_orange = np.array([20, 255, 255])
mask = cv2.inRange(hsv, lower_orange, upper_orange)

# Use Hough Circle to detect round objects (basketballs are round)
masked = cv2.bitwise_and(frame, frame, mask=mask)
gray = cv2.cvtColor(masked, cv2.COLOR_BGR2GRAY)
gray = cv2.GaussianBlur(gray, (9, 9), 2)
circles = cv2.HoughCircles(gray, cv2.HOUGH_GRADIENT, dp=1.2, minDist=30,
                           param1=100, param2=15, minRadius=5, maxRadius=40)

if circles is not None:
    circles = np.round(circles[0, :]).astype("int")
    x, y, r = circles[0]
    bbox = (x - r, y - r, 2 * r, 2 * r)
    print("Ball detected at:", bbox)
else:
    raise ValueError("Could not detect basketball automatically")

# Initialize tracker with ball position
tracker = cv2.TrackerKCF_create()
tracker.init(frame, bbox)

# Save to AVI first
intermediate_path = '../output/tracked_output.avi'
fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter(intermediate_path, fourcc, 20.0, (frame.shape[1], frame.shape[0]))

frame_count = 0
while True:
    success, frame = cap.read()
    if not success:
        break
    success, bbox = tracker.update(frame)
    if success:
        x, y, w, h = [int(v) for v in bbox]
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
    out.write(frame)
    frame_count += 1

cap.release()
out.release()
print("Tracking complete. Frames processed:", frame_count)

output_mp4 = '../output/tracked_output.mp4'
os.system(f"ffmpeg -y -i {intermediate_path} -vcodec libx264 -crf 23 -preset fast {output_mp4}")
print("MP4 video saved at:", output_mp4)

if os.path.exists(output_mp4):
    display(Video(output_mp4, embed=True, width=640, height=480))
else:
    print("MP4 output not found")


ValueError: Could not detect basketball automatically